In [ ]:
from dloader import MergeSet, TagDataset
from tokenizer import TagBPETokenizer

In [5]:
print("Training tokenizer..")
tagtok = TagBPETokenizer(vocab_size=10000, min_frequency=2)
data = MergeSet.from_file("../data/output/merged_tags.json")
extended_data = data.extend_with_synthetic(perm_limit=5, sub_array_count=5)
print(f"Total examples {len(extended_data)}")
tagtok.train(extended_data, "../ayo.json")
print("Done!")

Training tokenizer..
Total examples 131592
Tokenizer saved to ../ayo.json
Done!


In [6]:
print(tagtok.encode("1girl glasses handsup"))
print(tagtok.encode("glasses"))
print(tagtok.encode("dsa𓂀"))  # not in vocab
print(tagtok.encode_ids("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"))
print(len(tagtok.encode_ids(" ".join(data.real_examples[5000]))))

['1', 'girl', 'Ġglasses', 'Ġhand', 'su', 'p']
['glasses']
['d', 'sa', '[UNK]', 'ĵ', 'Ĥ', 'Ģ']
[33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 8317, 53, 54, 55, 56, 57, 58, 476, 64, 1023, 67, 68, 177, 71, 72, 73, 74, 407, 77, 78, 79, 321, 82, 83, 84, 3820, 87]
17


In [ ]:
from torch.utils.data import DataLoader

max_len_cut = 64
batch_size = 64

dataset = TagDataset(data.tags, tokenizer=tagtok, max_len_cut=max_len_cut)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F

In [109]:
buff = 100
emb = nn.Embedding(
    num_embeddings=tagtok.vocab_size + buff,
    embedding_dim=dataset.max_len_cut,
)
emb(torch.tensor([tagtok.vocab_size])).shape, emb(dataset[1]).shape

(torch.Size([1, 64]), torch.Size([64, 64]))

In [103]:
n_layers = 1
n_heads = 4
emb_dim = dataset.max_len_cut

transformer = nn.TransformerEncoder(
    encoder_layer=nn.TransformerEncoderLayer(
        d_model=emb_dim, nhead=n_heads, batch_first=True
    ),
    num_layers=n_layers,
    enable_nested_tensor=False
)

In [143]:
output_emb = 10
linproj = nn.Linear(2*emb_dim, output_emb)

In [145]:
def forward(x: torch.Tensor):
    ix = x                    # (B, I)
    x = emb(ix)               # (B, I, I)
    x = transformer(x)        # (B, I, I)
    x = torch.cat([
        x.mean(dim=1),        # (B, I) context
        x.max(dim=1).values   # (B, I) highlights
    ], dim=-1)                # (B, 2I)
    ox = linproj(x)           # (B, O)
    return ox

batch = torch.stack([
    dataset[0],
    dataset[8]
])

forward(batch).shape

torch.Size([2, 10])